# Phase 2 — Tag Track

카테고리별 핵심 태그가 조회수(log_views)에 미치는 영향 분석.  
계획: `docs/2/plan.md` · 점검 기록: `docs/2/Background.md`

## Step 0 — 셋업 & 데이터 검증
- 입력 로드 + 방어적 재중복 제거
- 누수 스코프 확인 (tags / category / log_views 만 사용)
- 카테고리별 n·median(views) 출력 → median Top3 = Gaming / Music / Film & Animation 재확인

In [3]:
import pandas as pd
import numpy as np
from collections import Counter
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# 시각화 스타일 (Part A run_final.py와 일관)
PRIMARY, ACCENT, GRAY = '#3B4FE4', '#1A7F5A', '#C8CDD6'
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

In [ ]:
CSV = '../../dataset/cleaned_USvideos.csv'
df = pd.read_csv(CSV)

# 방어적 재중복 제거 (집안 패턴: 최고 views 행 유지)
before = len(df)
df = (df.sort_values('views')
        .drop_duplicates('video_id', keep='last')
        .reset_index(drop=True))
print(f'rows: {before} -> {len(df)} (dedup)')

# 누수 스코프: tags / category / log_views 만 사용
required = ['video_id', 'category', 'log_views', 'tags', 'tag_cnt']
missing = [c for c in required if c not in df.columns]
assert not missing, f'missing columns: {missing}'
print('required columns OK')
print('shape:', df.shape)

rows: 6249 -> 6249 (dedup)
required columns OK
shape: (6249, 26)


In [5]:
# 카테고리별 n + median(views) — median Top3 재확인
cat_stats = (df.groupby('category')['views']
               .agg(n='count', median_views='median')
               .sort_values('median_views', ascending=False))
print(cat_stats.to_string())
print()

TARGET = ['Gaming', 'Music', 'Film & Animation']  # median Top3 (plan B-1)
top3_actual = cat_stats.head(3).index.tolist()
print('median Top3 (actual):', top3_actual)
print('target              :', TARGET)
print('match:', top3_actual == TARGET)
print()

present = [c for c in TARGET if c in cat_stats.index]
print('target category n:')
print(cat_stats.loc[present, 'n'].to_string())
if len(present) < len(TARGET):
    print('NOTE: missing target category name ->', set(TARGET) - set(present))

                          n  median_views
category                                 
Gaming                  101     1430025.0
Music                   792     1144372.5
Film & Animation        306     1080609.5
Comedy                  541      798810.0
Shows                     4      765522.0
Entertainment          1599      619443.0
Autos & Vehicles         65      451675.0
Science & Technology    369      448413.0
Pets & Animals          138      436488.5
Howto & Style           590      431936.5
People & Blogs          487      425636.0
Sports                  442      422192.0
Travel & Events          58      409661.5
Education               242      387893.0
News & Politics         501      182121.0
Nonprofits & Activism    14       36963.5

median Top3 (actual): ['Gaming', 'Music', 'Film & Animation']
target              : ['Gaming', 'Music', 'Film & Animation']
match: True

target category n:
category
Gaming              101
Music               792
Film & Animation    306
